<a href="https://colab.research.google.com/github/phishinqi/phishinqi.github.io/blob/main/Colab_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instrument-Agnostic Automatic Music Transcription (Colab Inference)

> ⚠️ **IMPORTANT — the first `Run all` stops once by design**
>
> The setup cell installs the locked native packages and automatically restarts the Colab Python session. This ends the first `Run all`; it is expected, not a failure.
>
> After Colab reconnects, continue from **2. Prepare Audio**. Alternatively, choose **Run all** again; setup recognizes the installed environment and will not restart a second time.

This notebook runs inference with [instrument-agnostic-amt](https://github.com/anime-song/instrument-agnostic-amt).
Dependencies are installed from the repository's `uv.lock`, and the pre-trained model is downloaded automatically from Hugging Face.

## 1. Setup Environment

Run the setup cell once to install the locked dependencies. Colab will restart its Python session automatically; after it reconnects, continue directly to step 2 without rerunning the setup cell. On the first run, **Run all** pauses at the restart; after reconnection, continue from step 2 or choose **Run all** again.

In [1]:
# @title Install dependencies
import hashlib
import json
import os
from pathlib import Path
import shlex
import subprocess
import sys
import time
import uuid

REPOSITORY_URL = "https://github.com/anime-song/instrument-agnostic-amt.git"
PROJECT_DIR = Path("/content") / Path(REPOSITORY_URL).stem
PYLOCK_PATH = Path("/content/pylock.iaamt-colab.toml")
SETUP_STATE_PATH = Path("/content/.iaamt-colab-setup.json")
if "_IAAMT_KERNEL_TOKEN" not in globals():
    _IAAMT_KERNEL_TOKEN = uuid.uuid4().hex
IAAMT_KERNEL_TOKEN = _IAAMT_KERNEL_TOKEN

def run_step(label, command):
    print(f"\n▶ {label}", flush=True)
    print(f"$ {shlex.join(command)}", flush=True)
    started_at = time.monotonic()
    process = subprocess.Popen(command)
    while True:
        try:
            return_code = process.wait(timeout=15)
            break
        except subprocess.TimeoutExpired:
            elapsed = int(time.monotonic() - started_at)
            print(f"… {label} still running ({elapsed}s elapsed)", flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    elapsed = int(time.monotonic() - started_at)
    print(f"✓ {label} complete ({elapsed}s)", flush=True)

# Use an absolute clone destination so rerunning this cell cannot create nested clones.
if not PROJECT_DIR.exists():
    run_step(
        "Cloning repository",
        ["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)],
    )
elif not (PROJECT_DIR / ".git").exists():
    raise RuntimeError(f"Existing path is not a Git checkout: {PROJECT_DIR}")
else:
    print(f"✓ Using existing checkout: {PROJECT_DIR}", flush=True)

os.chdir(PROJECT_DIR)
print(f"✓ Working directory: {PROJECT_DIR}", flush=True)
run_step(
    "Installing uv 0.8.17",
    [sys.executable, "-m", "pip", "install", "uv==0.8.17"],
)

lock_sha256 = hashlib.sha256((PROJECT_DIR / "uv.lock").read_bytes()).hexdigest()
try:
    setup_state = json.loads(SETUP_STATE_PATH.read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError):
    setup_state = {}

if setup_state.get("lock_sha256") != lock_sha256:
    # Keep each locked package on its recorded index, including PyTorch's cu130 wheels.
    run_step(
        "Exporting locked dependencies",
        [
            "uv", "export", "--frozen", "--no-dev", "--extra", "stem",
            "--format", "pylock.toml", "--preview-features", "pylock",
            "--quiet", "--output-file", str(PYLOCK_PATH),
        ],
    )
    run_step(
        "Installing locked dependencies",
        [
            "uv", "pip", "install", "--system", "--require-hashes",
            "--preview-features", "pylock", "-r", str(PYLOCK_PATH),
        ],
    )
    # Validate the exact public import in a fresh process before marking setup complete.
    run_step(
        "Validating the inference import",
        [
            sys.executable, "-c",
            "from infer_stem import run_stem_separated_transcription",
        ],
    )
    SETUP_STATE_PATH.write_text(
        json.dumps({
            "lock_sha256": lock_sha256,
            "install_kernel_token": IAAMT_KERNEL_TOKEN,
            "project_dir": str(PROJECT_DIR),
        }),
        encoding="utf-8",
    )
    print("\n✓ Dependencies installed successfully.", flush=True)
    print("↻ Restarting the Colab session once.", flush=True)
    print("After Colab reconnects, continue directly to step 2.", flush=True)
    get_ipython().kernel.do_shutdown(restart=True)
else:
    if setup_state.get("project_dir") != str(PROJECT_DIR):
        setup_state["project_dir"] = str(PROJECT_DIR)
        SETUP_STATE_PATH.write_text(json.dumps(setup_state), encoding="utf-8")
    # Refuse to continue until the process that replaced native packages is gone.
    if setup_state.get("install_kernel_token") == IAAMT_KERNEL_TOKEN:
        print("The package update is still loaded by this Python process. Restarting again.", flush=True)
        print("After Colab reconnects, continue directly to step 2.", flush=True)
        get_ipython().kernel.do_shutdown(restart=True)
    else:
        print("✓ Dependencies are already installed. Continue directly to step 2.", flush=True)

✓ Using existing checkout: /content/instrument-agnostic-amt
✓ Working directory: /content/instrument-agnostic-amt

▶ Installing uv 0.8.17
$ /usr/bin/python3 -m pip install uv==0.8.17
✓ Installing uv 0.8.17 complete (2s)
✓ Dependencies are already installed. Continue directly to step 2.


## 2. Prepare Audio

You can either upload a file or download one from a URL (e.g., YouTube).

In [2]:
# @title Upload audio file
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import sys

SETUP_STATE_PATH = Path("/content/.iaamt-colab-setup.json")
try:
    setup_state = json.loads(SETUP_STATE_PATH.read_text(encoding="utf-8"))
except (FileNotFoundError, json.JSONDecodeError) as exc:
    raise RuntimeError("Run the setup cell before preparing audio.") from exc

if "project_dir" in setup_state:
    stored_project_dir = setup_state["project_dir"]
    if not isinstance(stored_project_dir, str) or not stored_project_dir:
        raise RuntimeError("The saved repository path is invalid. Run the setup cell again.")
    PROJECT_DIR = Path(stored_project_dir)
else:
    project_candidates = sorted(
        lock_path.parent
        for lock_path in Path("/content").glob("*/uv.lock")
        if hashlib.sha256(lock_path.read_bytes()).hexdigest()
        == setup_state.get("lock_sha256")
    )
    if len(project_candidates) != 1:
        candidate_text = ", ".join(map(str, project_candidates)) or "none"
        raise RuntimeError(
            f"Could not identify the repository checkout ({candidate_text}). "
            "Run the setup cell again."
        )
    PROJECT_DIR = project_candidates[0]
    setup_state["project_dir"] = str(PROJECT_DIR)
    SETUP_STATE_PATH.write_text(json.dumps(setup_state), encoding="utf-8")

if not (PROJECT_DIR / "uv.lock").is_file():
    raise RuntimeError("The repository checkout is missing. Run the setup cell again.")

lock_sha256 = hashlib.sha256((PROJECT_DIR / "uv.lock").read_bytes()).hexdigest()
if setup_state.get("lock_sha256") != lock_sha256:
    raise RuntimeError("The lockfile changed. Run the setup cell again.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import numpy as np

installed_numpy_version = importlib.metadata.version("numpy")
if np.__version__ != installed_numpy_version:
    raise RuntimeError(
        "Colab is still using the previous NumPy version. Restart the session, then retry."
    )
import scipy.signal
import torch

IAAMT_SETUP_READY = True
print(
    f"Setup ready: NumPy {np.__version__}, SciPy signal, "
    f"PyTorch {torch.__version__}, CUDA available={torch.cuda.is_available()}"
)

from google.colab import files

uploaded = files.upload()
uploaded_name = next(iter(uploaded))
audio_path = str(Path(uploaded_name).resolve())
print(f"Uploaded: {audio_path}")

Setup ready: NumPy 2.5.2, SciPy signal, PyTorch 2.13.0+cu130, CUDA available=True


Saving 《牛来》片尾曲催泪钢琴改编，听懂的都哭了…….wav to 《牛来》片尾曲催泪钢琴改编，听懂的都哭了…….wav
Uploaded: /content/instrument-agnostic-amt/《牛来》片尾曲催泪钢琴改编，听懂的都哭了…….wav


## 3. Stem Separation -> Transcribe Each Stem -> Instrument Refinement -> Merge -> Velocity Prediction -> Beat/Chord/Key Prediction

This section builds on the `batch_process_unlabeled.py` flow inside Colab.
It separates the uploaded song into stems, transcribes each stem, optionally relabels the instrument of every note with the instrument refinement model, merges the per-stem MIDI files into one result, predicts per-note velocity dynamics using separated stem audio, and optionally predicts beat, chord, and key information using the beat_chord model (`best_beat_chord_key.pth`).
Drum stems use the dedicated experimental `drums` model, bass stems use the `bass_v2` model, guitar stems use the `guitar_v1_5` model, and other stems use the `other_v1_5` model.
Instrument refinement (`REFINE_INSTRUMENTS`) listens to each separated stem again and reassigns the instrument class of its notes, so a stem whose notes were labeled with the wrong instrument can be corrected before merging. Drum and vocal stems are always skipped: drums have no non-drum candidate classes, and separating `melody` from `vocal_harmony` is a matter of musical role rather than timbre, which this model cannot judge.

`DEVICE` defaults to `auto`, which picks CUDA → MPS → CPU; on Colab GPU runtimes this selects CUDA. `AMP` is enabled by default for the AMT transcription stage only, with `AMP_DTYPE` selecting the device default; velocity, refinement, and beat/chord always run in FP32, and AMP output can differ slightly from an FP32 run. `COMPILE_MODEL` (with `COMPILE_MODE`) enables opt-in regional `torch.compile`: only the Transformer blocks inside the AMT backbone are compiled, while CQT, convolutions, and decoding stay eager. The independent `COMPILE_VELOCITY` compiles the same Transformer regions of the velocity model; every window, including the trailing partial one, runs through the same compiled model. Both compile options are off by default, and the first processed window includes the compilation time.


In [3]:
# @title Prepare stem-separated transcription helpers
if not globals().get("IAAMT_SETUP_READY", False):
    raise RuntimeError(
        "Run the audio upload cell after Colab reconnects, then retry this cell."
    )
from infer_stem import run_stem_separated_transcription


In [7]:
# @title Run stem-separated transcription
OUTPUT_ROOT = "colab_outputs"  # @param {type:"string"}
STEM_SPLITTER_BATCH_SIZE = 1    # @param {type:"integer"}
WINDOW_BATCH_SIZE = 1  # @param {type:"integer"}
MAX_MIDI_MELODIC_INSTRUMENTS = 15  # @param {type:"integer"}
TRANSCRIBE_DRUM_STEMS = True  # @param {type:"boolean"}
REFINE_INSTRUMENTS = False  # @param {type:"boolean"}
REFINEMENT_CHECKPOINT = ""  # @param {type:"string"}
REFINEMENT_MODE = "cluster"  # @param ["cluster", "single"]
PREDICT_VELOCITY = True  # @param {type:"boolean"}
PREDICT_BEAT_CHORD = False  # @param {type:"boolean"}
CLEANUP_SEPARATED_STEMS = False  # @param {type:"boolean"}
MERGE_ONSET_MS = 50.0  # @param {type:"number"}
USE_ADTOF_TRANSCRIBER = False  # @param {type:"boolean"}
USE_STFT_VOCALIZER = False  # @param {type:"boolean"}
USE_TRANSKUN = False  # @param {type:"boolean"}
LOW_VRAM_MODE = False  # @param {type:"boolean"} Keep models in CPU RAM, move one to GPU per stem
NO_HALF = False  # @param {type:"boolean"} Disable fp16 (e.g. GTX 16-series): fp32 separation with halved chunks
DEVICE = "auto"  # @param ["auto", "cuda", "mps", "cpu"]
AMP = True  # @param {type:"boolean"}
AMP_DTYPE = "default"  # @param ["default", "fp16", "bf16"]
COMPILE_MODEL = False  # @param {type:"boolean"}
COMPILE_VELOCITY = False  # @param {type:"boolean"}
COMPILE_MODE = "default"  # @param ["default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs"]
SEMI_CRF_BACKEND = "auto"  # @param ["auto", "triton", "torch"]

if "audio_path" not in globals():
    raise RuntimeError("Please upload an audio file first.")

stem_pipeline_result = run_stem_separated_transcription(
    audio_path,
    checkpoint_path=None,
    output_root=OUTPUT_ROOT,
    stem_splitter_batch_size=STEM_SPLITTER_BATCH_SIZE,
    window_batch_size=WINDOW_BATCH_SIZE,
    max_midi_melodic_instruments=MAX_MIDI_MELODIC_INSTRUMENTS,
    transcribe_drum_stems=TRANSCRIBE_DRUM_STEMS,
    refine_instruments=REFINE_INSTRUMENTS,
    refinement_checkpoint_path=REFINEMENT_CHECKPOINT or None,
    refinement_mode=REFINEMENT_MODE,
    predict_velocity=PREDICT_VELOCITY,
    predict_beat_chord=PREDICT_BEAT_CHORD,
    cleanup_separated_stems=CLEANUP_SEPARATED_STEMS,
    merge_onset_ms=MERGE_ONSET_MS,
    device=DEVICE,
    amp=AMP,
    amp_dtype=None if AMP_DTYPE == "default" else AMP_DTYPE,
    compile_model=COMPILE_MODEL,
    compile_velocity=COMPILE_VELOCITY,
    compile_mode=COMPILE_MODE,
    semi_crf_backend=SEMI_CRF_BACKEND,
)
stem_pipeline_result

Checkpoint not found at checkpoints/best_model.pth. Downloading from Hugging Face...


100%|██████████| 53.5M/53.5M [00:00<00:00, 66.4MB/s]


Loading Separation model on cuda ...
[INFO] 重みファイルが見つかりません。Hugging Face から取得します: anime-song/stem-splitter/stem_splitter.pt@main


Loading AMT model (default) on cuda ...
Missing keys while loading checkpoint: ('head.interval_instrument_predictor.net.0.bias', 'head.interval_instrument_predictor.net.0.weight', 'head.interval_instrument_predictor.net.1.bias', 'head.interval_instrument_predictor.net.1.weight', 'head.interval_instrument_predictor.net.4.bias', 'head.interval_instrument_predictor.net.4.weight', 'head.slot_embedding.weight')
Separating stems for: 《牛来》片尾曲催泪钢琴改编，听懂的都哭了…….wav (batch_size=1)
Checkpoint not found at checkpoints/best_model_bass_v2.pth. Downloading from Hugging Face...


100%|██████████| 54.5M/54.5M [00:01<00:00, 47.8MB/s]


...

100%|██████████| 54.5M/54.5M [00:00<00:00, 69.9MB/s]


Loading AMT model (vocal_harmony_v1_5) on cuda ...
Transcribing stem: vocals (model=vocal_harmony_v1_5, instruments=melody,vocal_harmony,choir)
Predicting note velocities from separated stems...
Velocity checkpoint not found at checkpoints/best_velocity_model.pth. Downloading from Hugging Face...


100%|██████████| 52.4M/52.4M [00:01<00:00, 54.8MB/s]


Skipping velocity backbone for stems without notes: ['bass', 'drums', 'guitar', 'other', 'vocals']
Updated merged MIDI with predicted velocities: colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……/merged/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……_velocity.mid
Final Merged MIDI: colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……/merged/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……_velocity.mid


{'audio_path': '/content/instrument-agnostic-amt/《牛来》片尾曲催泪钢琴改编，听懂的都哭了…….wav',
 'output_root': 'colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……',
 'stem_dir': 'colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……/stems',
 'stem_midi_dir': 'colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……/stem_midis',
 'merged_midi_path': 'colab_outputs/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……/merged/《牛来》片尾曲催泪钢琴改编，听懂的都哭了……_velocity.mid',
 'stem_count': 6,
 'transcribed_stem_count': 6,
 'instruments_refined': False,
 'velocity_predicted': True,
 'beat_chord_predicted': False,
 'semi_crf_backend': 'triton'}

In [8]:
# @title Download stem-separated results
from google.colab import files
from pathlib import Path
import shutil

if "stem_pipeline_result" not in globals():
    print("Run the stem-separated transcription cell first.")
else:
    merged_midi_path = Path(stem_pipeline_result["merged_midi_path"])
    stem_midi_dir = Path(stem_pipeline_result["stem_midi_dir"])
    zip_base = stem_midi_dir.parent / f"{stem_midi_dir.name}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=stem_midi_dir))

    print(f"Downloading merged MIDI: {merged_midi_path}")
    files.download(str(merged_midi_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: Run Inference

In [ ]:
# @title Run Transcription
!python infer.py --audio "{audio_path}"

import os
midi_path = os.path.splitext(audio_path)[0] + ".mid"
if os.path.exists(midi_path):
    print(f"Success! MIDI saved to: {midi_path}")
else:
    print("Error: MIDI file was not generated.")

## Optional: Download Results

In [ ]:
# @title Download MIDI file
if os.path.exists(midi_path):
    files.download(midi_path)
else:
    print("No MIDI file to download.")